In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
# Week 9 — L2 Dates + Text + De-dup: unify keys, parse dates, clean price, dedup
import pandas as pd

raw = pd.DataFrame({
    'order_id':[1,2,3,3],
    'user':['  alice  ','ALICE','Bob','bob '],
    'date':['2024/01/02','01-03-2024','2024.01.04','2024-01-04'],
    'price':['$10','USD 12','13 USD',None]
})

# Normalize text keys for stable joins/grouping
raw['user_norm'] = raw['user'].str.strip().str.lower()

# Extract numeric price; coerce invalid to 0 then float
raw['price_num'] = (raw['price'].fillna('0')
    .str.replace(r'[^0-9\.]','', regex=True).astype(float))

# Robust date parsing; coerce malformed to NaT
raw['dt'] = pd.to_datetime(raw['date'], errors='coerce', dayfirst=False, infer_datetime_format=True)

# De-dup within (order_id, user_norm), keep first
clean = raw.drop_duplicates(subset=['order_id','user_norm'], keep='first')

# Weekly summary
clean['week'] = clean['dt'].dt.isocalendar().week
summary = (clean.groupby(['user_norm','week'])['price_num']
           .agg(total_spend='sum', orders='count').reset_index())

clean.to_csv('result_week9_clean_orders.csv', index=False)
summary.to_csv('result_week9_clean_summary.csv', index=False)


/tmp/ipykernel_47/3604404572.py:19: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  raw['dt'] = pd.to_datetime(raw['date'], errors='coerce', dayfirst=False, infer_datetime_format=True)
/tmp/ipykernel_47/3604404572.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean['week'] = clean['dt'].dt.isocalendar().week
